# Imports

In [58]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd

# Funções

In [59]:
# Get scores.json file from a path
def get_scores(path):
    scores_path = Path(path) / "scores.json"
    with open(scores_path, "r") as f:
        scores = json.load(f)

    # check if score has relevancy and factuality columns
    # if not search for score and score_secondary columns and rename them to relevancy and factuality
    if "factuality" not in scores and "score_secondary" in scores:
        scores["factuality"] = scores.pop("score_secondary")
        # move to beginning of the dictionary
        scores = {"factuality": scores.pop("factuality"), **scores}
    if "relevancy" not in scores and "score" in scores:
        scores["relevancy"] = scores.pop("score")
        # move to beginning of the dictionary
        scores = {"relevancy": scores.pop("relevancy"), **scores}
    
    # create overall as first item in scores dict
    # overall = (relevancy + factuality) / 2
    scores["overall"] = (scores["relevancy"] + scores["factuality"]) / 2

    # move overall to the first position in the dictionary
    scores = {"overall": scores.pop("overall"), **scores}

    return scores

# Get all scores from a list of paths
def get_all_scores(paths):
    all_scores = []
    for path in paths:
        scores = get_scores(path)
        all_scores.append(scores)
    return all_scores

# Convert a list of scores to a DataFrame
def scores_to_df(paths):
    # Get all scores
    scores_list = get_all_scores(paths)

    # Get model names from paths
    model_names = [Path(path).name for path in paths]

    # Create a DataFrame from the scores and model names
    df = pd.DataFrame(scores_list)
    df["model"] = model_names

    # Make model the first column
    cols = df.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    df = df[cols]

    return df

In [60]:
baseline = "../../../artifacts/results/baseline"

In [61]:
get_scores(baseline)

{'overall': 0.34275,
 'relevancy': 0.5344,
 'factuality': 0.1511,
 'bert': 0.6086,
 'rouge': 0.271,
 'similarity': 0.9324,
 'bleurt': 0.3257,
 'medcat': 0.1722,
 'align': 0.1299}

In [62]:
paths = [baseline]
get_all_scores(paths)

[{'overall': 0.34275,
  'relevancy': 0.5344,
  'factuality': 0.1511,
  'bert': 0.6086,
  'rouge': 0.271,
  'similarity': 0.9324,
  'bleurt': 0.3257,
  'medcat': 0.1722,
  'align': 0.1299}]

In [63]:
scores_to_df(paths)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.34275,0.5344,0.1511,0.6086,0.271,0.9324,0.3257,0.1722,0.1299


# MedGemma

## MedGemma Puro

In [64]:
results_mg0_sp  = "../../../artifacts/results/rag_simple_prompt_20260310_113904"
results_mg0_fs  = "../../../artifacts/results/rag_few_shot_20260312_103549"
results_mg0_rag = "../../../artifacts/results/rag_base_20260223_204535"

In [65]:
# create df with all results
paths = [baseline, results_mg0_sp, results_mg0_fs, results_mg0_rag]
df = scores_to_df(paths)
df

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
1,rag_simple_prompt_20260310_113904,0.303262,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209
2,rag_few_shot_20260312_103549,0.304593,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517
3,rag_base_20260223_204535,0.326805,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757


## MedGemma Fine Tuned

In [66]:
results_mg0_ft_sp  = "../../../artifacts/results/simple_prompt_fine_tune_full_20260408_084221"
results_mg0_ft_fs  = "../../../artifacts/results/few_shot_fine_tuning_full_20260409_093749"
results_mg0_ft_rag = "../../../artifacts/results/rag_fine_tune_full_20260406_134649"

In [67]:
paths_ft = [baseline, results_mg0_ft_sp, results_mg0_ft_fs, results_mg0_ft_rag]
scores_to_df(paths_ft)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
1,simple_prompt_fine_tune_full_20260408_084221,0.324787,0.495646,0.153928,0.580538,0.219084,0.892597,0.290366,0.147047,0.160809
2,few_shot_fine_tuning_full_20260409_093749,0.304068,0.478473,0.129664,0.582002,0.205553,0.835759,0.290578,0.112550,0.146777
3,rag_fine_tune_full_20260406_134649,0.306847,0.482912,0.130783,0.575279,0.196777,0.867951,0.291639,0.139918,0.121648


In [68]:
## all meg gemma
paths_mg_puro = [
    baseline, 
    results_mg0_sp, 
    results_mg0_fs, 
    results_mg0_rag, 
    results_mg0_ft_sp, 
    results_mg0_ft_fs, 
    results_mg0_ft_rag
]
scores_to_df(paths_mg_puro).sort_values("overall", ascending=False)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
3,rag_base_20260223_204535,0.326805,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757
4,simple_prompt_fine_tune_full_20260408_084221,0.324787,0.495646,0.153928,0.580538,0.219084,0.892597,0.290366,0.147047,0.160809
6,rag_fine_tune_full_20260406_134649,0.306847,0.482912,0.130783,0.575279,0.196777,0.867951,0.291639,0.139918,0.121648
2,rag_few_shot_20260312_103549,0.304593,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517
5,few_shot_fine_tuning_full_20260409_093749,0.304068,0.478473,0.129664,0.582002,0.205553,0.835759,0.290578,0.112550,0.146777
1,rag_simple_prompt_20260310_113904,0.303262,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209


# MedGemma 1.5B

## MedGemma 1.5B Puro

## MedGemma 1.5B Fine Tuned

In [69]:
results_mg_ft_sp  = "../../../artifacts/results/simple_prompt_med15_4B_finetuned_20260422_102940"
results_mg_ft_fs = "../../../artifacts/results/few_shot_med15_4B_finetuned_20260426_105810"
results_mg_ft_rag = "../../../artifacts/results/rag_prompt_med15_4B_finetuned_20260426_203118"

In [71]:
paths_mg_15B_ft = [
    baseline, 
    results_mg_ft_sp,
    results_mg_ft_fs, 
    results_mg_ft_rag
]
scores_to_df(paths_mg_15B_ft).sort_values("overall", ascending=False)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
1,simple_prompt_med15_4B_finetuned_20260422_102940,0.330277,0.511993,0.148561,0.587698,0.228274,0.924946,0.307056,0.154150,0.142972
2,few_shot_med15_4B_finetuned_20260426_105810,0.294369,0.459144,0.129594,0.577464,0.156472,0.828219,0.274420,0.112758,0.146429
3,rag_prompt_med15_4B_finetuned_20260426_203118,0.293824,0.461448,0.126201,0.571731,0.157490,0.839488,0.277082,0.123668,0.128733


## MedGemma 1.5B Fine Tuned with RAG

In [76]:
#results_mg_ftr_sp  =
results_mg_ftr_fs  = "../../../artifacts/results/few_shot_med15_4B_finetuned_with_rag_20260427_093030"
results_mg_ftr_rag = "../../../artifacts/results/rag_prompt_med15_4B_finetuned_with_rag_20260425_081439"

In [77]:
paths_mg_15B_ftr = [
    baseline, 
    #results_mg_ftr_sp,
    results_mg_ftr_fs, 
    results_mg_ftr_rag
]
scores_to_df(paths_mg_15B_ftr).sort_values("overall", ascending=False)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
2,rag_prompt_med15_4B_finetuned_with_rag_2026042...,0.306882,0.451215,0.162548,0.579309,0.159991,0.805535,0.260026,0.139341,0.185755
1,few_shot_med15_4B_finetuned_with_rag_20260427_...,0.288108,0.441427,0.134789,0.580538,0.156926,0.763650,0.264593,0.117738,0.151839


## all med gemma

In [54]:
all_mg_results = [
    baseline,
    results_mg0_sp,
    results_mg0_fs,
    results_mg0_rag,
    results_mg0_ft_sp,
    results_mg0_ft_fs,
    results_mg0_ft_rag,
    results_mg_ft_sp,
    results_mg_ft_rag,
]
scores_to_df(all_mg_results).sort_values("overall", ascending=False)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
7,simple_prompt_med15_4B_finetuned_20260422_102940,0.330277,0.511993,0.148561,0.587698,0.228274,0.924946,0.307056,0.154150,0.142972
3,rag_base_20260223_204535,0.326805,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757
4,simple_prompt_fine_tune_full_20260408_084221,0.324787,0.495646,0.153928,0.580538,0.219084,0.892597,0.290366,0.147047,0.160809
8,rag_prompt_med15_4B_finetuned_with_rag_2026042...,0.306882,0.451215,0.162548,0.579309,0.159991,0.805535,0.260026,0.139341,0.185755
6,rag_fine_tune_full_20260406_134649,0.306847,0.482912,0.130783,0.575279,0.196777,0.867951,0.291639,0.139918,0.121648
2,rag_few_shot_20260312_103549,0.304593,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517
5,few_shot_fine_tuning_full_20260409_093749,0.304068,0.478473,0.129664,0.582002,0.205553,0.835759,0.290578,0.112550,0.146777
1,rag_simple_prompt_20260310_113904,0.303262,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209


# Qwen

## Qwen Puro

In [56]:
results_qwen_sp   = "../../../artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239"
#results_qwen_fs   = ""
results_qwen_rag  = "../../../artifacts/results/rag_prompt_qwen3_VL_4B_20260423_010343"

In [57]:
results_qwen = [
    baseline,
    results_qwen_sp,
    results_qwen_rag
]
scores_to_df(results_qwen).sort_values("overall", ascending=False)

,model,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,0.342750,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
2,rag_prompt_qwen3_VL_4B_20260423_010343,0.331478,0.519108,0.143848,0.600307,0.217038,0.931983,0.327104,0.152968,0.134729
1,simple_prompt_qwen3_VL_4B_20260422_192239,0.325250,0.486996,0.163504,0.580141,0.182586,0.878392,0.306865,0.129695,0.197314


# Qwen Fine Tuning

In [ ]:
#results_qwen_sp   = "artifacts/results/simple_prompt_qwen3_VL_4B_20260422_192239"
#results_qwen_fs   = ""
#results_qwen_rag  = "artifacts/results/rag_prompt_qwen3_VL_4B_20260423_010343"